In [3]:
#seq2seq英译法案例
#案例步骤：
#1.数据预处理：读取数据，构建词汇表，转换为索引
#2.模型定义：定义编码器和解码器
#3.训练模型：定义损失函数和优化器，训练模型
#4.评估模型：测试模型性能，进行翻译

import re #正则表达式模块
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import time

import random #随机数生成器
import matplotlib.pyplot as plt

device=torch.device("cuda" if torch.cuda.is_available() else "cpu") #设备适配
SOS_token=0 #设置开始标记和结束标记
EOS_token=1

max_length=10 #设置最大句子长度
path='./data/eng-fra-v2.txt'




In [23]:
#数据预处理
#读取文档数据，构建词汇表和索引映射
eng_sentences=[] #英语句子列表
fra_sentences=[]
with open(path,'r',encoding='utf-8') as f:
    sentences=f.read().strip().split('\n')
    print(sentences[:5]) #打印前5行数据
    for sentence in sentences:
        eng,fra=sentence.split('\t')
        eng_sentences.append(eng)
        fra_sentences.append(fra)
    print(eng_sentences[:5])
    print(len(fra_sentences))

#构建两个句子列表的字典对应
eng_fra_dict={}
for i in range(len(eng_sentences)):
    eng_fra_dict.update({eng_sentences[i]:fra_sentences[i]})
# print(eng_fra_dict)

#构建词汇表和索引映射
eng_words=[]
fra_words=[]
for sentence in eng_sentences:
    eng_words.extend(sentence.split(' '))
print(eng_words[:5])

for sentence in fra_sentences:
    fra_words.extend(sentence.split(' '))
print(fra_words[:5])

eng_vocab=list(set(eng_words)) #英语词汇表
fra_vocab=list(set(fra_words))
print(len(eng_vocab))
print(len(fra_vocab))

eng_word_idx={'SOS':SOS_token,'EOS':EOS_token} #英语词汇索引映射
fra_word_idx={'SOS':SOS_token,'EOS':EOS_token} #法语词汇
i=2
j=2
for word in eng_vocab:
    eng_word_idx[word]=i
    i+=1
for word in fra_vocab:
    fra_word_idx[word]=j
    j+=1
# print(fra_word_idx)
# print(fra_word_idx.items())

eng_idx_word={i:w for w,i in eng_word_idx.items()}
fra_idx_word={i:w for w,i in fra_word_idx.items()}
print(eng_idx_word)

eng_seq_idx=[]
fra_seq_idx=[]
#将句子转换为索引序列
for sentence in eng_sentences:
    tem=[eng_word_idx[w] for w in sentence.split(' ')]
    eng_seq_idx.append(tem)
print(eng_seq_idx[:5])

for sentence in fra_sentences:
    tem=[fra_word_idx[w] for w in sentence.split(' ')]
    fra_seq_idx.append(tem)
print(fra_seq_idx[:5])

['i m .\tj ai ans .', 'i m ok .\tje vais bien .', 'i m ok .\tca va .', 'i m fat .\tje suis gras .', 'i m fat .\tje suis gros .']
['i m .', 'i m ok .', 'i m ok .', 'i m fat .', 'i m fat .']
63594
['i', 'm', '.', 'i', 'm']
['j', 'ai', 'ans', '.', 'je']
2801
4343
{0: 'SOS', 1: 'EOS', 2: 'answering', 3: 'low', 4: 'amazed', 5: 'matters', 6: 'sensing', 7: 'are', 8: 'competitors', 9: 'embassy', 10: 'quite', 11: 'gardener', 12: 'ordering', 13: 'dealing', 14: 'fifties', 15: 'cat', 16: 'shopping', 17: 'smoker', 18: 'charmed', 19: 'beard', 20: 'joy', 21: 'hair', 22: 'upstairs', 23: 'shivering', 24: 'morons', 25: 'distantly', 26: 'introverted', 27: 'sales', 28: 'demented', 29: 'missing', 30: 'bartender', 31: 'nature', 32: 'homeschooled', 33: 'susceptible', 34: 'floor', 35: 'result', 36: 'earthquakes', 37: 'scapegoat', 38: 'absent', 39: 'discussing', 40: 'sociology', 41: 'experiences', 42: 'set', 43: 'thieves', 44: 'crime', 45: 'lazy', 46: 'energy', 47: 'explain', 48: 'tv', 49: 'familiar', 50: 'get